In [1]:
!pip install transformers datasets
import json
import pandas as pd
import numpy as np
import transformers
import sklearn
import datasets
from transformers import pipeline
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
from transformers import DataCollatorWithPadding
from transformers import TrainingArguments
from transformers import AutoModelForSequenceClassification
from transformers import Trainer
from sklearn.metrics import classification_report
from transformers import set_seed
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score

df_train = pd.read_json('project_training.json')
df_val = pd.read_json('project_validation.json') #hyperopt

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 49.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.0/469.0 KB 28.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 70.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 KB 14.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.9/132.9 KB 8.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 KB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.2/212.2 KB 10.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.6/264.6 KB 9.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.2/199.2 KB 24.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.8/158.8 KB 22.1 MB/s eta 0:00:00
     ━━━━━━━━━━━

In [2]:
df_train["climate"] = df_train["climate"].replace({"yes": 0, "no": 1})
df_train["sentiment"] = df_train["sentiment"].replace({"opportunity": 0, "neutral": 1, "risk": 2})
df_train["specificity"] = df_train["specificity"].replace({"spec": 0, "non": 1})
df_train["commitment"] = df_train["commitment"].replace({"yes": 0, "no": 1})
df_train

,text,climate,sentiment,commitment,specificity
0,The accelerator programs have sub-portfolios o...,0,1.0,1.0,1.0
1,"Also by means of BNDES Finem, we offer credit ...",1,NaN,NaN,NaN
2,Climate change Climate change exposes UPM to v...,0,2.0,1.0,1.0
3,Several tools and methodologies aimed at asses...,0,2.0,0.0,1.0
4,We worked with the UK government to accelerate...,0,0.0,0.0,0.0
...,...,...,...,...,...
395,"At the beginning of 2019, VINCI Airports signe...",1,NaN,NaN,NaN
396,We have also signed up to the Partnership for ...,0,2.0,0.0,1.0
397,Suzano also is involved and spearheads externa...,0,1.0,0.0,1.0
398,Risks to the Group’s reputation Risks include ...,1,NaN,NaN,NaN


In [3]:
df_val["climate"] = df_val["climate"].replace({"yes": 0, "no": 1})
df_val["sentiment"] = df_val["sentiment"].replace({"opportunity": 0, "neutral": 1, "risk": 2})
df_val["specificity"] = df_val["specificity"].replace({"spec": 0, "non": 1})
df_val["commitment"] = df_val["commitment"].replace({"yes": 0, "no": 1})
df_val

,text,climate,sentiment,commitment,specificity
0,"Except as required by law, the Bank does not u...",1,NaN,NaN,NaN
1,"In terms of thermal coal, the Group has set an...",0,1.0,0.0,0.0
2,The engagement is driving improved conversatio...,0,1.0,0.0,1.0
3,The tool does this by tracking the generation ...,0,1.0,1.0,1.0
4,We also support and advise companies in the La...,0,0.0,0.0,0.0
...,...,...,...,...,...
395,"JetBlue drives safety management in many ways,...",1,NaN,NaN,NaN
396,10 Representative Concentration Pathways (RCP)...,0,1.0,1.0,1.0
397,Building on our history of energy efficiency i...,0,1.0,0.0,0.0
398,"In 2019, PME redesigned the passively managed ...",0,2.0,0.0,0.0


In [4]:
df = df_train.append(df_val)

In [5]:
checkpoint = "climatebert/distilroberta-base-climate-f"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize(batch):
    tokens = tokenizer(batch['text'], truncation=True, max_length=512) # question: finetune? 10,50,100, 512
    tokens['label'] = batch['label']
    return tokens

In [6]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


# Climate

In [7]:
df_train = df[["text", "climate"]]
df_train.rename(columns={"climate": "label"}, inplace = 1)
df_train

/usr/local/lib/python3.9/dist-packages/pandas/core/frame.py:5039: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  return super().rename(


,text,label
0,The accelerator programs have sub-portfolios o...,0
1,"Also by means of BNDES Finem, we offer credit ...",1
2,Climate change Climate change exposes UPM to v...,0
3,Several tools and methodologies aimed at asses...,0
4,We worked with the UK government to accelerate...,0
...,...,...
395,"JetBlue drives safety management in many ways,...",1
396,10 Representative Concentration Pathways (RCP)...,0
397,Building on our history of energy efficiency i...,0
398,"In 2019, PME redesigned the passively managed ...",0


In [8]:
n=5
kf = KFold(n_splits=n, random_state=0, shuffle=True)

In [9]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [10]:
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import accuracy_score
results = []
accuracy = []

for train_index, val_index in kf.split(df):
    # splitting Dataframe
    train_df = df_train.iloc[train_index]
    val_df = df_train.iloc[val_index]

    train_dataset = datasets.Dataset.from_pandas(train_df)
    val_dataset = datasets.Dataset.from_pandas(val_df)

    train_dataset = train_dataset.map(tokenize, remove_columns=["text"])
    val_dataset = val_dataset.map(tokenize, remove_columns=["text"])

    # Defining Model
    set_seed(0)
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
    model = model.to(device)
    
    # train the model
    training_args = TrainingArguments(
    output_dir="/content",
    per_device_train_batch_size = 5,
    num_train_epochs = 3,
    )

    trainer = Trainer(
        model,
        training_args,
        train_dataset=train_dataset,
        tokenizer=tokenizer
    )

    trainer.train()

    pred = trainer.predict(val_dataset)

    results.append(precision_recall_fscore_support(pred.label_ids, pred.predictions.argmax(1), average='macro'))
    accuracy.append(accuracy_score(pred.label_ids, pred.predictions.argmax(1)))

Map:   0%|          | 0/640 [00:00<?, ? examples/s]

Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Some weights of the model checkpoint at climatebert/distilroberta-base-climate-f were not used when initializing RobertaForSequenceClassification: ['lm_head.layer_norm.bias', 'lm_head.layer_norm.weight', 'lm_head.dense.bias', 'lm_head.dense.weight', 'lm_head.bias']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at climatebert/distilroberta-base-climate-f and are newly initialized: ['classifier.out_proj.weight', 'classifier.dense.weight', 'classifie

Step,Training Loss




Training completed. Do not forget to share your model on huggingface.co/models =)


The following columns in the test set don't have a corresponding argument in `RobertaForSequenceClassification.forward` and have been ignored: __index_level_0__. If __index_level_0__ are not expected by `RobertaForSequenceClassification.forward`,  you can safely ignore this message.
***** Running Prediction *****
  Num examples = 160
  Batch size = 8


...

Map:   0%|          | 0/640 [00:00<?, ? examples/s]

Map:   0%|          | 0/160 [00:00<?, ? examples/s]

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--climatebert--distilroberta-base-climate-f/snapshots/d04f8afaceb89c882a110ada795bf580627c13c4/config.json
Model config RobertaConfig {
  "_name_or_path": "climatebert/distilroberta-base-climate-f",
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 6,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "torch_dtype": "float32",
  "transformers_version": "4.26.1",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50500
}

loading weights file pytorch_model.bin from

Step,Training Loss




Training completed. Do not forget to share your model on huggingface.co/models =)


The following columns in the test set don't have a corresponding argument in `RobertaForSequenceClassification.forward` and have been ignored: __index_level_0__. If __index_level_0__ are not expected by `RobertaForSequenceClassification.forward`,  you can safely ignore this message.
***** Running Prediction *****
  Num examples = 160
  Batch size = 8


In [11]:
print("results",results)
print(accuracy)

results [(0.9846153846153847, 0.9411764705882353, 0.9609375, None), (0.9172077922077921, 0.929406850459482, 0.9231560891938251, None), (0.9718137254901962, 0.9562962962962963, 0.9638526997514874, None), (0.98828125, 0.9571428571428571, 0.9716830865435668, None), (0.9930555555555556, 0.9444444444444444, 0.9670917317976142, None)]
[0.975, 0.95625, 0.98125, 0.98125, 0.9875]


In [12]:
avg_precision = np.mean([results[0][0], results[1][0], results[2][0], results[3][0], results[4][0]])
avg_recall = np.mean([results[0][1], results[1][1], results[2][1], results[3][1], results[4][1]])
avg_f = np.mean([results[0][2], results[1][2], results[2][2], results[3][2], results[4][2]])

In [13]:
print(avg_precision)
print(avg_recall)
print(avg_f)
print(np.mean(accuracy))

0.9709947415737856
0.945693383786263
0.9573442214572987
0.97625


#sentiment

In [14]:
df_train = df[["text", "sentiment"]].dropna().reset_index(drop=True)
df_train.rename(columns={"sentiment": "label"}, inplace = 1)
df_train["label"] = df_train["label"].astype(int)
df_train

,text,label
0,The accelerator programs have sub-portfolios o...,1
1,Climate change Climate change exposes UPM to v...,2
2,Several tools and methodologies aimed at asses...,2
3,We worked with the UK government to accelerate...,0
4,"(changes in consumption behavior, failure of i...",2
...,...,...
656,Investec group including Investec Asset Manage...,2
657,10 Representative Concentration Pathways (RCP)...,1
658,Building on our history of energy efficiency i...,1
659,"In 2019, PME redesigned the passively managed ...",2


In [15]:
n=5
kf = KFold(n_splits=n, random_state=0, shuffle=True)

In [16]:
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import accuracy_score
results = []
accuracy = []

for train_index, val_index in kf.split(df_train):
    # splitting Dataframe (dataset not included)
    train_df = df_train.iloc[train_index]
    val_df = df_train.iloc[val_index]

    train_dataset = datasets.Dataset.from_pandas(train_df)
    val_dataset = datasets.Dataset.from_pandas(val_df)

    train_dataset = train_dataset.map(tokenize, remove_columns=["text"])
    val_dataset = val_dataset.map(tokenize, remove_columns=["text"])

    # Defining Model
    set_seed(0)
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=3)
    model = model.to(device)
    
    # train the model
    training_args = TrainingArguments(
    output_dir="/content",
    per_device_train_batch_size = 5,
    num_train_epochs = 3,
    )

    trainer = Trainer(
        model,
        training_args,
        train_dataset=train_dataset,
        tokenizer=tokenizer
    )

    trainer.train()

    pred = trainer.predict(val_dataset)

    results.append(precision_recall_fscore_support(pred.label_ids, pred.predictions.argmax(1), average='macro'))
    accuracy.append(accuracy_score(pred.label_ids, pred.predictions.argmax(1)))


Map:   0%|          | 0/528 [00:00<?, ? examples/s]

Map:   0%|          | 0/133 [00:00<?, ? examples/s]

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--climatebert--distilroberta-base-climate-f/snapshots/d04f8afaceb89c882a110ada795bf580627c13c4/config.json
Model config RobertaConfig {
  "_name_or_path": "climatebert/distilroberta-base-climate-f",
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2
  },
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 6,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "torch_dtype": "float

Step,Training Loss




Training completed. Do not forget to share your model on huggingface.co/models =)


The following columns in the test set don't have a corresponding argument in `RobertaForSequenceClassification.forward` and have been ignored: __index_level_0__. If __index_level_0__ are not expected by `RobertaForSequenceClassification.forward`,  you can safely ignore this message.
***** Running Prediction *****
  Num examples = 133
  Batch size = 8


...

Map:   0%|          | 0/529 [00:00<?, ? examples/s]

Map:   0%|          | 0/132 [00:00<?, ? examples/s]

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--climatebert--distilroberta-base-climate-f/snapshots/d04f8afaceb89c882a110ada795bf580627c13c4/config.json
Model config RobertaConfig {
  "_name_or_path": "climatebert/distilroberta-base-climate-f",
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2
  },
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 6,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "torch_dtype": "float

Step,Training Loss




Training completed. Do not forget to share your model on huggingface.co/models =)


The following columns in the test set don't have a corresponding argument in `RobertaForSequenceClassification.forward` and have been ignored: __index_level_0__. If __index_level_0__ are not expected by `RobertaForSequenceClassification.forward`,  you can safely ignore this message.
***** Running Prediction *****
  Num examples = 132
  Batch size = 8


In [17]:
avg_precision = np.mean([results[0][0], results[1][0], results[2][0], results[3][0], results[4][0]])
avg_recall = np.mean([results[0][1], results[1][1], results[2][1], results[3][1], results[4][1]])
avg_f = np.mean([results[0][2], results[1][2], results[2][2], results[3][2], results[4][2]])

print(avg_precision)
print(avg_recall)
print(avg_f)
print(np.mean(accuracy))

0.8401832618630506
0.8402904246312068
0.8387455656035392
0.8396331738437002


#Commitment

In [18]:
df_train = df[["text", "commitment"]].dropna().reset_index(drop=True)
df_train.rename(columns={"commitment": "label"}, inplace = 1)
df_train["label"] = df_train["label"].astype(int)
df_train

,text,label
0,The accelerator programs have sub-portfolios o...,1
1,Climate change Climate change exposes UPM to v...,1
2,Several tools and methodologies aimed at asses...,0
3,We worked with the UK government to accelerate...,0
4,"(changes in consumption behavior, failure of i...",1
...,...,...
656,Investec group including Investec Asset Manage...,1
657,10 Representative Concentration Pathways (RCP)...,1
658,Building on our history of energy efficiency i...,0
659,"In 2019, PME redesigned the passively managed ...",0


In [19]:
n=5
kf = KFold(n_splits=n, random_state=0, shuffle=True)

from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import accuracy_score
results = []
accuracy = []

for train_index, val_index in kf.split(df_train):
    # splitting Dataframe (dataset not included)
    train_df = df_train.iloc[train_index]
    val_df = df_train.iloc[val_index]

    train_dataset = datasets.Dataset.from_pandas(train_df)
    val_dataset = datasets.Dataset.from_pandas(val_df)

    train_dataset = train_dataset.map(tokenize, remove_columns=["text"])
    val_dataset = val_dataset.map(tokenize, remove_columns=["text"])

    # Defining Model
    set_seed(0)
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
    model = model.to(device)
    
    # train the model
    training_args = TrainingArguments(
    output_dir="/content",
    per_device_train_batch_size = 5,
    num_train_epochs = 3,
    )

    trainer = Trainer(
        model,
        training_args,
        train_dataset=train_dataset,
        tokenizer=tokenizer
    )

    trainer.train()

    pred = trainer.predict(val_dataset)

    results.append(precision_recall_fscore_support(pred.label_ids, pred.predictions.argmax(1), average='macro'))
    accuracy.append(accuracy_score(pred.label_ids, pred.predictions.argmax(1)))

Map:   0%|          | 0/528 [00:00<?, ? examples/s]

Map:   0%|          | 0/133 [00:00<?, ? examples/s]

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--climatebert--distilroberta-base-climate-f/snapshots/d04f8afaceb89c882a110ada795bf580627c13c4/config.json
Model config RobertaConfig {
  "_name_or_path": "climatebert/distilroberta-base-climate-f",
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 6,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "torch_dtype": "float32",
  "transformers_version": "4.26.1",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50500
}

loading weights file pytorch_model.bin from

Step,Training Loss




Training completed. Do not forget to share your model on huggingface.co/models =)


The following columns in the test set don't have a corresponding argument in `RobertaForSequenceClassification.forward` and have been ignored: __index_level_0__. If __index_level_0__ are not expected by `RobertaForSequenceClassification.forward`,  you can safely ignore this message.
***** Running Prediction *****
  Num examples = 133
  Batch size = 8


...

Map:   0%|          | 0/529 [00:00<?, ? examples/s]

Map:   0%|          | 0/132 [00:00<?, ? examples/s]

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--climatebert--distilroberta-base-climate-f/snapshots/d04f8afaceb89c882a110ada795bf580627c13c4/config.json
Model config RobertaConfig {
  "_name_or_path": "climatebert/distilroberta-base-climate-f",
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 6,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "torch_dtype": "float32",
  "transformers_version": "4.26.1",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50500
}

loading weights file pytorch_model.bin from

Step,Training Loss




Training completed. Do not forget to share your model on huggingface.co/models =)


The following columns in the test set don't have a corresponding argument in `RobertaForSequenceClassification.forward` and have been ignored: __index_level_0__. If __index_level_0__ are not expected by `RobertaForSequenceClassification.forward`,  you can safely ignore this message.
***** Running Prediction *****
  Num examples = 132
  Batch size = 8


In [20]:
avg_precision = np.mean([results[0][0], results[1][0], results[2][0], results[3][0], results[4][0]])
avg_recall = np.mean([results[0][1], results[1][1], results[2][1], results[3][1], results[4][1]])
avg_f = np.mean([results[0][2], results[1][2], results[2][2], results[3][2], results[4][2]])

print(avg_precision)
print(avg_recall)
print(avg_f)
print(np.mean(accuracy))

0.8822366886192364
0.8832575248871176
0.8815530772660864
0.8835383914331283


#specificity

In [21]:
df_train = df[["text", "specificity"]].dropna().reset_index(drop=True)
df_train.rename(columns={"specificity": "label"}, inplace = 1)
df_train["label"] = df_train["label"].astype(int)
df_train

,text,label
0,The accelerator programs have sub-portfolios o...,1
1,Climate change Climate change exposes UPM to v...,1
2,Several tools and methodologies aimed at asses...,1
3,We worked with the UK government to accelerate...,0
4,"(changes in consumption behavior, failure of i...",1
...,...,...
656,Investec group including Investec Asset Manage...,1
657,10 Representative Concentration Pathways (RCP)...,1
658,Building on our history of energy efficiency i...,0
659,"In 2019, PME redesigned the passively managed ...",0


In [22]:
n=5
kf = KFold(n_splits=n, random_state=0, shuffle=True)

from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import accuracy_score
results = []
accuracy = []

for train_index, val_index in kf.split(df_train):
    # splitting Dataframe (dataset not included)
    train_df = df_train.iloc[train_index]
    val_df = df_train.iloc[val_index]

    train_dataset = datasets.Dataset.from_pandas(train_df)
    val_dataset = datasets.Dataset.from_pandas(val_df)

    train_dataset = train_dataset.map(tokenize, remove_columns=["text"])
    val_dataset = val_dataset.map(tokenize, remove_columns=["text"])

    # Defining Model
    set_seed(0)
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
    model = model.to(device)
    
    # train the model
    training_args = TrainingArguments(
    output_dir="/content",
    per_device_train_batch_size = 5,
    num_train_epochs = 3,
    )

    trainer = Trainer(
        model,
        training_args,
        train_dataset=train_dataset,
        tokenizer=tokenizer
    )

    trainer.train()

    pred = trainer.predict(val_dataset)

    results.append(precision_recall_fscore_support(pred.label_ids, pred.predictions.argmax(1), average='macro'))
    accuracy.append(accuracy_score(pred.label_ids, pred.predictions.argmax(1)))




Map:   0%|          | 0/528 [00:00<?, ? examples/s]

Map:   0%|          | 0/133 [00:00<?, ? examples/s]

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--climatebert--distilroberta-base-climate-f/snapshots/d04f8afaceb89c882a110ada795bf580627c13c4/config.json
Model config RobertaConfig {
  "_name_or_path": "climatebert/distilroberta-base-climate-f",
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 6,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "torch_dtype": "float32",
  "transformers_version": "4.26.1",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50500
}

loading weights file pytorch_model.bin from

Step,Training Loss




Training completed. Do not forget to share your model on huggingface.co/models =)


The following columns in the test set don't have a corresponding argument in `RobertaForSequenceClassification.forward` and have been ignored: __index_level_0__. If __index_level_0__ are not expected by `RobertaForSequenceClassification.forward`,  you can safely ignore this message.
***** Running Prediction *****
  Num examples = 133
  Batch size = 8


...

Map:   0%|          | 0/529 [00:00<?, ? examples/s]

Map:   0%|          | 0/132 [00:00<?, ? examples/s]

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--climatebert--distilroberta-base-climate-f/snapshots/d04f8afaceb89c882a110ada795bf580627c13c4/config.json
Model config RobertaConfig {
  "_name_or_path": "climatebert/distilroberta-base-climate-f",
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 6,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "torch_dtype": "float32",
  "transformers_version": "4.26.1",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50500
}

loading weights file pytorch_model.bin from

Step,Training Loss




Training completed. Do not forget to share your model on huggingface.co/models =)


The following columns in the test set don't have a corresponding argument in `RobertaForSequenceClassification.forward` and have been ignored: __index_level_0__. If __index_level_0__ are not expected by `RobertaForSequenceClassification.forward`,  you can safely ignore this message.
***** Running Prediction *****
  Num examples = 132
  Batch size = 8


In [23]:
avg_precision = np.mean([results[0][0], results[1][0], results[2][0], results[3][0], results[4][0]])
avg_recall = np.mean([results[0][1], results[1][1], results[2][1], results[3][1], results[4][1]])
avg_f = np.mean([results[0][2], results[1][2], results[2][2], results[3][2], results[4][2]])

print(avg_precision)
print(avg_recall)
print(avg_f)
print(np.mean(accuracy))

0.8685355166329007
0.8647059981451454
0.8647297577262222
0.8698792435634541
